Libraries

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sb
import requests
import time
import os
import dask.dataframe as dd
from IPython.display import display

pd.set_option('display.max_columns', None)

In [3]:
# Definir rango de fechas para Enero 2021

start_date = '2021-01-01'
end_date = '2021-01-31'
# end_date = '2022-12-31'

date_range = pd.date_range(start=start_date, end=end_date)
# print(date_range)

In [4]:
# Importar Enero 2021 - forma 1

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()
df = []

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df.append(pd.read_csv(url))

# Combinar los DataFrames diarios en uno solo.
enero = pd.concat(df, ignore_index=True)

# Calcular tiempo de carga
load_time_1 = time.time() - start_time
print(f"Tiempo de carga: {load_time_1:.2f} s")

Tiempo de carga: 2.33 s


In [5]:
## Instalar aiohttp (necesario para fsspec HTTPFileSystem usado por Dask/pandas)
#%pip install aiohttp -q
#
#import aiohttp

In [ ]:
# Importar Enero 2021 - forma 2 (con Dask)

# Definir lista de DF's y tiempo inicial de carga
start_time = time.time()
df2 = []

for single_date in date_range:
    url = f'https://raw.githubusercontent.com/CSSEGISandData/COVID-19/refs/heads/master/csse_covid_19_data/csse_covid_19_daily_reports/{single_date.strftime("%m-%d-%Y")}.csv'
    df2.append(dd.read_csv(url, dtype={'Admin2': 'object'}))

# Combinar los DataFrames diarios en uno solo.
enero2 = dd.concat(df2, ignore_index=True)

# Calcular tiempo de carga
load_time_2 = time.time() - start_time
print(f"Tiempo de carga con Dask: {load_time_2:.2f} s")

Tiempo de carga con Dask: 1.97 s


In [33]:
# 1. Cargar y visualizar los primeros 5 registros

enero = enero2
enero.head()

,FIPS,Admin2,Province_State,Country_Region,Last_Update,Lat,Long_,Confirmed,Deaths,Recovered,Active,Combined_Key,Incident_Rate,Case_Fatality_Ratio,last_update
0,NaN,<NA>,<NA>,Afghanistan,2021-01-02 05:22:33,33.93911,67.709953,52513,2201,41727,8585,Afghanistan,134.896578,4.191343,2021-01-02 05:22:33
1,NaN,<NA>,<NA>,Albania,2021-01-02 05:22:33,41.15330,20.168300,58316,1181,33634,23501,Albania,2026.409062,2.025173,2021-01-02 05:22:33
2,NaN,<NA>,<NA>,Algeria,2021-01-02 05:22:33,28.03390,1.659600,99897,2762,67395,29740,Algeria,227.809861,2.764848,2021-01-02 05:22:33
3,NaN,<NA>,<NA>,Andorra,2021-01-02 05:22:33,42.50630,1.521800,8117,84,7463,570,Andorra,10505.403482,1.034865,2021-01-02 05:22:33
4,NaN,<NA>,<NA>,Angola,2021-01-02 05:22:33,-11.20270,17.873900,17568,405,11146,6017,Angola,53.452981,2.305328,2021-01-02 05:22:33


In [8]:
# 2. Mostrar el número total de filas y columnas del DataFrame.

print('Filas en total: ', len(enero))
print('Columnas en total: ', len(enero.columns))

Filas en total:  124398
Columnas en total:  14


In [9]:
# 3. Describir los tipos de datos (dtypes) y convertir las columnas necesarias (por ejemplo,
# fechas).

enero.dtypes

FIPS                           float64
Admin2                 string[pyarrow]
Province_State         string[pyarrow]
Country_Region         string[pyarrow]
Last_Update            string[pyarrow]
Lat                            float64
Long_                          float64
Confirmed                        int64
Deaths                           int64
Recovered                        int64
Active                           int64
Combined_Key           string[pyarrow]
Incident_Rate                  float64
Case_Fatality_Ratio            float64
dtype: object

In [ ]:
# 3

# Formatear la columna Last_Update a tipo datetime

enero['Last_Update'] = pd.to_datetime(enero['Last_Update'], format = 'ISO8601')

enero.dtypes

FIPS                           float64
Admin2                 string[pyarrow]
Province_State         string[pyarrow]
Country_Region         string[pyarrow]
Last_Update            string[pyarrow]
Lat                            float64
Long_                          float64
Confirmed                      float64
Deaths                         float64
Recovered                      float64
Active                         float64
Combined_Key           string[pyarrow]
Incident_Rate                  float64
Case_Fatality_Ratio            float64
last_update             datetime64[ns]
dtype: object


In [31]:
# 3

# Convertir columnas categóricas a tipo 'category' para optimizar memoria (Dask)
cols_to_cat = [c for c in ['Province_State', 'Country_Region', 'Combined_Key'] if c in enero2.columns]
if cols_to_cat:
    try:
        # Dask tiene una operación 'categorize' que es eficiente para convertir múltiples particiones
        enero2 = enero2.categorize(columns=cols_to_cat)
    except Exception:
        for c in cols_to_cat:
            enero2[c] = enero2[c].astype('category')

# Calcular uso de memoria aproximado (requiere computar la suma)
memoria_antes = enero2.memory_usage(deep=True).sum().compute()
memoria_antes_mb = memoria_antes / (1024 * 1024)
print(f"Uso de memoria (estimado): {memoria_antes_mb:.2f} MB")

Uso de memoria (estimado): 23.52 MB


In [32]:
# Ejemplo: cuando necesites una operación de pandas específica, convierte a pandas con .compute()
# Esto traerá los datos a memoria — úsalo sólo cuando quepa en RAM o para subconjuntos.
enero_pdf = enero2.compute()
print('DataFrame pandas obtenido con .compute():', type(enero_pdf))
# Ejemplo de operación específica de pandas: crear columna active_cases y ver memoria local
enero_pdf['active_cases'] = enero_pdf['Confirmed'] - enero_pdf['Deaths'] - enero_pdf['Recovered']
memoria_pdf_mb = enero_pdf.memory_usage(deep=True).sum() / (1024*1024)
print(f"Memoria del DataFrame pandas en memoria: {memoria_pdf_mb:.2f} MB")

DataFrame pandas obtenido con .compute(): <class 'pandas.core.frame.DataFrame'>
Memoria del DataFrame pandas en memoria: 17.23 MB


In [ ]:
enero.head(0)

ValueError: Mismatched dtypes found in `pd.read_csv`/`pd.read_table`.

+--------+--------+----------+
| Column | Found  | Expected |
+--------+--------+----------+
| Admin2 | object | float64  |
+--------+--------+----------+

The following columns also raised exceptions on conversion:

- Admin2
  ValueError("could not convert string to float: 'Autauga'")

Usually this is due to dask's dtype inference failing, and
*may* be fixed by specifying dtypes manually by adding:

dtype={'Admin2': 'object'}

to the call to `read_csv`/`read_table`.

In [ ]:
# 4. Detectar y mostrar valores nulos o faltantes por columna.

display(enero.isnull().sum())

FIPS                   23166
Admin2                 23011
Province_State          5529
Country_Region             0
Last_Update                0
Lat                     2776
Long_                   2776
Confirmed                  0
Deaths                     0
Recovered                  0
Active                     0
Combined_Key               0
Incident_Rate           2776
Case_Fatality_Ratio     1484
dtype: int64

In [ ]:
# 5. Eliminar columnas irrelevantes (por ejemplo, códigos FIPS o coordenadas si no se usarán).

enero = enero.drop(columns=['FIPS', 'Admin2', 'Lat', 'Long_', 'Combined_Key'])
enero.head(1)

,Province_State,Country_Region,Last_Update,Confirmed,Deaths,Recovered,Active,Incident_Rate,Case_Fatality_Ratio
0,NaN,Afghanistan,2021-01-02 05:22:33,52513,2201,41727,8585,134.896578,4.191343


In [ ]:
# 6. Estandarizar nombres de columnas (usar formato snake_case).

enero.columns = enero.columns.str.lower().str.replace(' ', '_')
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
0,NaN,Afghanistan,2021-01-02 05:22:33,52513,2201,41727,8585,134.896578,4.191343


In [ ]:
# 7. Homogeneizar nombres de países (ej. “US” → “United States”).

enero['country_region'] = enero['country_region'].replace({'US': 'United States'})
enero[enero['country_region'] == 'United States'].head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
648,Alabama,United States,2021-01-02 05:22:33,4239,50,0,4189,7587.391935,1.179523


In [ ]:
# 8. Convertir la columna last_update al formato YYYY-MM-DD (día preciso)
# Asegurar datetime y mantener sólo fecha (YYYY-MM-DD)
enero['last_update'] = pd.to_datetime(enero['last_update'], errors='coerce').dt.date
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio
0,NaN,Afghanistan,2021-01-02 05:22:33,52513,2201,41727,8585,134.896578,4.191343


In [ ]:
# 9. Crear una columna active_cases = Confirmed - Deaths - Recovered.

enero['active_cases'] = (enero['confirmed'] - enero['deaths'] - enero['recovered'])
enero.head(1)

,province_state,country_region,last_update,confirmed,deaths,recovered,active,incident_rate,case_fatality_ratio,active_cases
0,NaN,Afghanistan,2021-01-02 05:22:33,52513,2201,41727,8585,134.896578,4.191343,8585


In [ ]:
# 10. Guardar el DataFrame limpio como covid_clean_enero2020.csv e indicar su tamaño en MB.

enero.to_csv('covid_clean_enero2020.csv')

In [ ]:
# 10

import os

file_size = os.path.getsize('covid_clean_enero2020.csv') / (1024 * 1024)  # Convertir a MB
print(f'El tamaño del archivo covid_clean_enero2020.csv es: {file_size:.2f} MB')

El tamaño del archivo covid_clean_enero2020.csv es: 12.27 MB
